# PRG330 – AI Impact on Jobs and Salaries (2020–2025)
**Team 4:** Aryan Ghimire • Sulav Ghimire • Krish Naga • Dippan Poudel

**Data note:** the supplied CSV contains records from 2020–2025. The project PDF describes a 2020–2026 study, but no 2026 records are present in the supplied CSV.


## Module 1 – Variables & Fundamentals

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_FILE = "ai_jobs_salaries_clean.csv"
df = pd.read_csv(DATA_FILE)

job_title: str = "ML Engineer"
work_year: int = 2025
salary_usd: float = 175000.0
is_outlier: bool = False
monthly_comp = salary_usd / 12
print(f"Example monthly pay: ${monthly_comp:,.2f}")
print("Dataset shape:", df.shape)
print("Years:", sorted(df["work_year"].unique()))
print("Duplicate rows:", df.duplicated().sum())
print(df.isna().sum())


## Module 2 – Conditionals & Logic

In [ ]:
sample_salary = 175000
if sample_salary >= 200000:
    tier = "Tier 1: Top Compensation"
elif sample_salary >= 120000:
    tier = "Tier 2: Competitive Market"
else:
    tier = "Tier 3: Emerging / Entry"
print("Salary tier:", tier)

def classify_work_mode(remote_ratio):
    if remote_ratio == 100:
        return "Remote"
    elif remote_ratio == 50:
        return "Hybrid"
    else:
        return "On-site"

df["work_mode_check"] = df["remote_ratio"].apply(classify_work_mode)

q1 = np.percentile(df["salary_in_usd"], 25)
q3 = np.percentile(df["salary_in_usd"], 75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
df["calculated_outlier"] = (df["salary_in_usd"] < lower_bound) | (df["salary_in_usd"] > upper_bound)
print("IQR bounds:", lower_bound, upper_bound)
print("Calculated outliers:", int(df["calculated_outlier"].sum()))


## Module 3 – Loops & Batch Iteration

In [ ]:
annual_growth = {}
for year in sorted(df["work_year"].unique()):
    subset = df.loc[df["work_year"] == year, "salary_in_usd"]
    annual_growth[year] = subset.mean()

for year, value in annual_growth.items():
    print(f"{year}: ${value:,.2f}")

role_counts = {}
for role in df["role_family"].unique():
    role_counts[role] = int((df["role_family"] == role).sum())

for role, count in sorted(role_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"{role}: {count:,}")


## Module 4 – Functions & Modularity

In [ ]:
def calculate_salary_stats(data_list):
    sorted_data = sorted(data_list)
    return {"mean": np.mean(sorted_data), "median": np.median(sorted_data), "count": len(sorted_data)}

def detect_outliers(series):
    q1 = np.percentile(series, 25)
    q3 = np.percentile(series, 75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

def role_analyzer(data, role):
    subset = data.loc[data["role_family"] == role, "salary_in_usd"]
    if len(subset) == 0:
        return None
    return {"role": role, "count": len(subset), "mean": subset.mean(), "median": subset.median()}

print(calculate_salary_stats(df["salary_in_usd"].tolist()))
print("Outlier boundaries:", detect_outliers(df["salary_in_usd"]))
print(role_analyzer(df, "ML Engineer"))


## Module 5 – Pandas & NumPy Data Engines

In [ ]:
year_summary = df.groupby("work_year")["salary_in_usd"].agg(records="count", mean="mean", median="median").reset_index()
role_summary = df.groupby("role_family")["salary_in_usd"].agg(records="count", mean="mean", median="median").reset_index().sort_values("median", ascending=False)
experience_summary = df.groupby("experience_level_label")["salary_in_usd"].agg(records="count", mean="mean", median="median").reset_index()
workmode_summary = df.groupby("work_mode")["salary_in_usd"].agg(records="count", mean="mean", median="median").reset_index()

display(year_summary.round(2))
display(role_summary.round(2))
display(experience_summary.round(2))
display(workmode_summary.round(2))

ai_ml_roles = ["AI Engineer", "ML Engineer", "AI Architect", "NLP", "Computer Vision", "Research Scientist"]
classic_data_roles = ["Data Analyst", "Data Engineer", "Data Scientist", "Analytics Manager"]
df["analysis_group"] = np.select([df["role_family"].isin(ai_ml_roles), df["role_family"].isin(classic_data_roles)], ["AI/ML","Classic Data"], default="Other")
premium_summary = df[df["analysis_group"].isin(["AI/ML","Classic Data"])].groupby("analysis_group")["salary_in_usd"].agg(records="count", mean="mean", median="median").reset_index()
display(premium_summary.round(2))


## Module 6 – Matplotlib & Seaborn Visuals

In [ ]:
sns.set_theme(style="whitegrid")

sns.histplot(df["salary_in_usd"], bins=60, kde=True)
plt.title("Salary Distribution in Global Tech Jobs")
plt.xlabel("Salary (USD)")
plt.show()

plt.figure(figsize=(10,6))
plt.plot(year_summary["work_year"], year_summary["median"], marker="o", label="Median")
plt.plot(year_summary["work_year"], year_summary["mean"], marker="o", label="Mean")
plt.title("Annual Salary Trend")
plt.xlabel("Year"); plt.ylabel("Salary (USD)"); plt.legend(); plt.show()

plt.figure(figsize=(10,7))
role_plot = role_summary.sort_values("median")
plt.barh(role_plot["role_family"], role_plot["median"])
plt.title("Median Salary by Role Family")
plt.xlabel("Median Salary (USD)"); plt.show()

plt.figure(figsize=(10,6))
sns.boxplot(data=df, x="experience_level_label", y="salary_in_usd", order=["Entry-level","Mid-level","Senior-level","Executive-level"])
plt.ylim(0, df["salary_in_usd"].quantile(.98))
plt.title("Salary Distribution by Experience Level"); plt.show()

sns.barplot(data=workmode_summary, x="work_mode", y="median")
plt.title("Median Salary by Work Mode"); plt.ylabel("Median Salary (USD)"); plt.show()

sns.heatmap(df[["work_year","salary","salary_in_usd","remote_ratio"]].corr(), annot=True, fmt=".2f", center=0)
plt.title("Correlation Matrix"); plt.show()

sns.barplot(data=premium_summary, x="analysis_group", y="median")
plt.title("AI/ML vs Classic Data: Median Salary"); plt.ylabel("Median Salary (USD)"); plt.show()


# Final Findings
- **71,913 rows × 17 columns**
- Actual CSV coverage: **2020–2025**
- **40,540** missing `isco_group_hint` values; other columns have no missing values
- **0 duplicate rows**
- Overall mean salary: **$151,161**
- Overall median salary: **$138,750**
- Highest median role family: **AI Architect ($195,000)**
- Lowest median listed role family: **Data Analyst ($99,450)**
- AI/ML median salary premium over the defined Classic Data group: **41.36%**
